# Notebook 03 - SFT Without Adaption

This is the raw-data SFT condition. It trains the base model on `data/processed/kakugo_raw_sft.jsonl`.

Notebook 04 mirrors this notebook. The only intended difference is the training data path.

**Files read:**
- [`../configs/tinker_sft_raw.yaml`](../configs/tinker_sft_raw.yaml) - base model, renderer, training hyperparameters, input path, and output path for the raw-data SFT run.
- [`../data/processed/kakugo_raw_sft.jsonl`](../data/processed/kakugo_raw_sft.jsonl) - chat-format raw SFT examples prepared in Notebook 01.

**Files written:**
- [`../results/models/sft_raw/`](../results/models/sft_raw/) - Tinker logs, metrics, checkpoints, and final model/sampler references for the raw-data SFT run.

In [4]:
import asyncio
import json
import logging
import os
import sys
import warnings
from pathlib import Path

import nest_asyncio
from huggingface_hub import login
from tinker_cookbook.supervised.data import FromConversationFileBuilder
from tinker_cookbook.supervised.train import Config, main
from tinker_cookbook.supervised.types import ChatDatasetBuilderCommonConfig

nest_asyncio.apply()
warnings.filterwarnings('ignore', module='tinker_cookbook')

In [5]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [6]:
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml, require_file

load_env()
raw_sft_config = load_yaml(ROOT / 'configs/tinker_sft_raw.yaml')

require_file(ROOT / raw_sft_config['data_path'], 'Run Notebook 01 before launching raw-data SFT.')
ensure_dir(ROOT / raw_sft_config['output_dir'])

if not os.environ.get('TINKER_API_KEY'):
    raise RuntimeError('Missing TINKER_TOKEN or TINKER_API_KEY. Add it to .env before launching training.')

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Review the training config

This is the exact raw-data SFT configuration that will be sent to Tinker in the training cell below.

In [7]:
print(json.dumps(raw_sft_config, indent=2))

{
  "run_name": "sft_raw",
  "base_model": "Qwen/Qwen3-8B-Base",
  "renderer_name": "role_colon",
  "data_path": "data/processed/kakugo_raw_sft.jsonl",
  "output_dir": "results/models/sft_raw",
  "learning_rate": "2e-4",
  "num_epochs": 3,
  "batch_size": 4,
  "lora_rank": 16,
  "max_length": 2048,
  "train_on_what": "last_assistant_message",
  "save_every": 50
}


## Build the Tinker SFT config

The dataset builder reads the chat-format JSONL created in Notebook 01. `train_on_what='last_assistant_message'` means the supervised loss is applied to the assistant response, not the user prompt.

In [8]:
renderer_name = raw_sft_config['renderer_name']

common_config = ChatDatasetBuilderCommonConfig(
    model_name_for_tokenizer=raw_sft_config['base_model'],
    renderer_name=renderer_name,
    max_length=int(raw_sft_config['max_length']),
    batch_size=int(raw_sft_config['batch_size']),
    train_on_what=raw_sft_config['train_on_what'],
)

dataset_builder = FromConversationFileBuilder(
    file_path=str(ROOT / raw_sft_config['data_path']),
    common_config=common_config,
)

sft_config = Config(
    log_path=str(ROOT / raw_sft_config['output_dir']),
    model_name=raw_sft_config['base_model'],
    dataset_builder=dataset_builder,
    learning_rate=float(raw_sft_config['learning_rate']),
    lora_rank=int(raw_sft_config['lora_rank']),
    num_epochs=int(raw_sft_config['num_epochs']),
)

print('\nSFT Config:')
print(f"  Run:           {raw_sft_config['run_name']}")
print(f"  Model:         {sft_config.model_name}")
print(f"  Renderer:      {renderer_name}")
print(f"  Data:          {ROOT / raw_sft_config['data_path']}")
print(f"  Output:        {sft_config.log_path}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  LoRA rank:     {sft_config.lora_rank}")
print(f"  Epochs:        {sft_config.num_epochs}")


SFT Config:
  Run:           sft_raw
  Model:         Qwen/Qwen3-8B-Base
  Renderer:      role_colon
  Data:          /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_raw_sft.jsonl
  Output:        /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_raw
  Learning rate: 0.0002
  LoRA rank:     16
  Epochs:        3


## Launch training

This cell starts the real Tinker SFT job. If credentials, package imports, data paths, or Tinker access are wrong, the cell should fail loudly so the issue is visible.

In [9]:
print('=' * 50)
print('Starting raw-data SFT training...')
print('Watch train_mean_nll; it should generally decrease across training.\n')

logging.getLogger('asyncio').setLevel(logging.CRITICAL)
result = asyncio.get_event_loop().run_until_complete(main(sft_config))

print('\nTraining complete!')
print(f'Result: {result}')

Starting raw-data SFT training...
Watch train_mean_nll; it should generally decrease across training.



root:680 [INFO] Command line invocation: /Users/vivianamarquez/anaconda3/envs/adaption-kirundi-sft/lib/python3.11/site-packages/ipykernel_launcher.py -f /Users/vivianamarquez/Library/Jupyter/runtime/kernel-21fad726-20f4-4cb9-a0ab-e020f555a561.json
tinker_cookbook.utils.ml_log:618 [INFO] Logging to: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_raw
tinker.lib.public_interfaces.service_client:75 [INFO] ServiceClient initialized for session 24bb570e-46c0-5a26-bb0e-11e4b49f5eb8
tinker.lib.public_interfaces.service_client:159 [INFO] TrainingClient initialized for model 24bb570e-46c0-5a26-bb0e-11e4b49f5eb8:train:0


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 146              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000005 │
│ num_loss_tokens      │ 1583     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2403     │
│ progress             │ 0.980000 │
│ time/finish_batch    │ 1.095021 │
│ time/get_batch       │ 0.006007 │
│ time/step            │ 1.094668 │
│ time/submit_batch    │ 0.007152 │
│ train_mean_nll       │ 1.567387 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Ibiharurup vy'ishirahamwe Medecins Sans Frontiere birenga kure ku bitangwa na reta harimwo n'abana barenga 700.
Inama nshingamateka y'u Burundi yemeje integuro y'amafaranga leta izokoresha mu mwaka uza wa 2018.
Icyegeranyo gishya cya guverinoma y'u rwanda gishinja abategetsi b'Ubufaransa gufasha abateguye jenoside.
Doug Jones yatsi

tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 147              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000004 │
│ num_loss_tokens      │ 1000     │
│ num_sequences        │ 4        │
│ num_tokens           │ 1903     │
│ progress             │ 0.986667 │
│ time/finish_batch    │ 1.260918 │
│ time/get_batch       │ 0.004749 │
│ time/step            │ 1.260468 │
│ time/submit_batch    │ 0.005715 │
│ train_mean_nll       │ 1.315527 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Ndashaka kumenya uburyo nabona ibikoresho biramba, bicuruzwa mu gihugu, kandi bitangiza ibidukikije. Wanyobora aho nabikura, ibiciro byabyo, n'uburyo bwo kubikoresha neza kugira ngo tugire uruhare mu kurengera ibidukikije?

Assistant: Ndagutahura umwuga w’ugura ibikoresho biramba, bicuruzwa imbere y’u Rwanda, bitangiza ibidukikije.

tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 148              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000003 │
│ num_loss_tokens      │ 1204     │
│ num_sequences        │ 4        │
│ num_tokens           │ 2504     │
│ progress             │ 0.993333 │
│ time/finish_batch    │ 0.896785 │
│ time/get_batch       │ 0.003549 │
│ time/step            │ 0.896455 │
│ time/submit_batch    │ 0.004481 │
│ train_mean_nll       │ 1.627368 │
└──────────────────────┴──────────┘
tinker_cookbook.utils.ml_log:206 [INFO] Wrote metrics to /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_raw/metrics.jsonl


tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 149              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000001 │
│ num_loss_tokens      │ 1259     │
│ num_sequences        │ 4        │
│ num_tokens           │ 1633     │
│ progress             │ 1.000000 │
│ time/finish_batch    │ 1.169436 │
│ time/step            │ 1.169132 │
│ train_mean_nll       │ 1.592053 │
└──────────────────────┴──────────┘
tinker_cookbook.checkpoint_utils:466 [INFO] Saved checkpoints: {'state_path': 'tinker://24bb570e-46c0-5a26-bb0e-11e4b49f5eb8:train:0/weights/final', 'sampler_path': 'tinker://24bb570e-46c0-5a26-bb0e-11e4b49f5eb8:train:0/sampler_weights/final'}
tinker_cookbook.supervised.train:518 [INFO] Training completed successfully



Training complete!
Result: None
